# Modul 03: Datenstrukturen, Quellen und Qualität

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Datenstrukturen, Quellen und Qualität  
    **Erwarteter Schwierigkeitsgrad:** Grundlagen bis leicht fortgeschritten  
    **Orientierungszeit:** etwa 80 bis 110 Minuten

    ## Überblick

    Sie strukturieren Beobachtungen, Merkmale und Zielwerte, laden dieselben Informationen aus mehreren lokalen Formaten und erstellen einen kompakten, wiederholbaren Validierungsbericht einschließlich Datenwörterbuch und Quellenmetadaten.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_03A_20260823.ipynb`
- `ML Für Anfänger - Record_Module_03B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Beobachtungen, Merkmale und Zielwerte in tabellarischen Daten beschreiben.
- Eingabematrix X und Zielvektor y mit korrekten Formen erstellen.
- Datentypen, Wertebereiche, Einheiten und Array-Formen dokumentieren.
- Kleine Datensätze aus Listen, CSV, JSON und mehreren Quellen laden.
- Spaltennamen, Datentypen, Fehlwerte, Duplikate und Wertebereiche prüfen.
- Einen kompakten Validierungsbericht für die ML-Vorbereitung erstellen.

    ## Bewertete Fähigkeiten

    - Zeilen-, Spalten- und Batchbedeutung sicher unterscheiden
- StringIO, JSON und Tabellenkonstruktion verwenden
- X/y sauber trennen und Zielwerttyp einordnen
- Datenwörterbuch, Quellenmetadaten und Qualitätsbericht erstellen

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from io import StringIO
import json

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

csv_source_a = '''machine_id,temperature_c,vibration_mm_s,shift,failure
M-101,62.5,2.1,Tag,0
M-102,68.0,2.8,Nacht,0
M-103,,5.7,Tag,1
M-104,71.5,3.2,Nacht,0
'''

json_source_b = '''[
  {"machine_id": "M-105", "temperature_c": 75.0, "vibration_mm_s": 6.4, "shift": "Nacht", "failure": 1},
  {"machine_id": "M-106", "temperature_c": 64.0, "vibration_mm_s": 2.5, "shift": "Tag", "failure": 0},
  {"machine_id": "M-106", "temperature_c": 64.0, "vibration_mm_s": 2.5, "shift": "Tag", "failure": 0},
  {"machine_id": "M-107", "temperature_c": 130.0, "vibration_mm_s": -1.0, "shift": "Abend", "failure": 0}
]'''

source_metadata = pd.DataFrame(
    {
        "source": ["sensor_export_A.csv", "maintenance_snapshot_B.json"],
        "version": ["2026-08-01", "2026-08-02"],
        "license_or_permission": ["interne Lehrdaten", "interne Lehrdaten"],
        "notes": ["Temperaturwert kann fehlen", "enthält bewusst Duplikat und Grenzfälle"],
    }
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Beobachtungen, Merkmale und Zielwert strukturieren

    Erstellen Sie aus der vorgegebenen Liste einen DataFrame. Trennen Sie anschließend:

- `X` mit den numerischen Merkmalen `temperature_c` und `vibration_mm_s`,
- `y` mit dem Zielwert `failure`.

Geben Sie Formen, Dimensionen und Datentypen aus. Erzeugen Sie zusätzlich einen Batch mit den ersten zwei Beobachtungen und erklären Sie seine Form.

> **Hinweis:** Die erste Dimension zählt Beobachtungen, die zweite Dimension zählt Merkmale.

In [ ]:
records = [
    {"machine_id": "M-001", "temperature_c": 61.0, "vibration_mm_s": 2.0, "failure": 0},
    {"machine_id": "M-002", "temperature_c": 79.0, "vibration_mm_s": 6.1, "failure": 1},
    {"machine_id": "M-003", "temperature_c": 66.5, "vibration_mm_s": 2.9, "failure": 0},
    {"machine_id": "M-004", "temperature_c": 82.0, "vibration_mm_s": 6.8, "failure": 1},
]

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Beobachtungen, Merkmale und Zielwert strukturieren
#
# Ziel dieser Codezelle:
# Erstellen Sie aus der vorgegebenen Liste einen DataFrame. Trennen Sie
# anschließend: - X mit den numerischen Merkmalen temperaturec und vibrationmms, - y
# mit dem Zielwert failure. Geben Sie Formen, Dimensionen und Date...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

records = [
    {"machine_id": "M-001", "temperature_c": 61.0, "vibration_mm_s": 2.0, "failure": 0},
    {"machine_id": "M-002", "temperature_c": 79.0, "vibration_mm_s": 6.1, "failure": 1},
    {"machine_id": "M-003", "temperature_c": 66.5, "vibration_mm_s": 2.9, "failure": 0},
    {"machine_id": "M-004", "temperature_c": 82.0, "vibration_mm_s": 6.8, "failure": 1},
]

# Jede Dictionary-Zeile wird zu einer Beobachtung im DataFrame.
machine_data = pd.DataFrame(records)

# Die Eingabematrix enthält nur Merkmale, die später dem Modell zur
# Verfügung stehen dürfen. machine_id ist hier eine Kennung, kein Messwert.
X = machine_data[["temperature_c", "vibration_mm_s"]].to_numpy(
    dtype=np.float64
)

# Der Zielvektor enthält genau einen bekannten Zielwert pro Beobachtung.
y = machine_data["failure"].to_numpy(dtype=np.int64)

# Die ersten zwei Zeilen bilden einen Batch mit zwei Beobachtungen und
# denselben zwei Merkmalen wie die vollständige Eingabematrix.
first_batch = X[:2]

print(machine_data)
print("X-Form:", X.shape, "X-Dimensionen:", X.ndim, "X-Typ:", X.dtype)
print("y-Form:", y.shape, "y-Dimensionen:", y.ndim, "y-Typ:", y.dtype)
print("Batch-Form:", first_batch.shape)
print(first_batch)

### Reflexion zu Aufgabe 1

`X` besitzt die Form `(4, 2)`: vier Beobachtungen und zwei Merkmale. `y` besitzt die Form `(4,)`, weil zu jeder Beobachtung genau ein Klassenlabel gehört. Der Batch `(2, 2)` enthält zwei Beobachtungen mit je zwei Merkmalen. Eine häufige Fehlerquelle ist, den Zielwert versehentlich in `X` zu belassen oder eine Kennung ohne fachliche Prüfung als numerisches Merkmal zu behandeln.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: CSV und JSON laden und Herkunft erhalten

    Laden Sie `csv_source_a` mit `StringIO` und `json_source_b` mit `json.loads` beziehungsweise `pd.DataFrame`. Ergänzen Sie in jeder Tabelle eine Spalte `source_file`, vereinheitlichen Sie die Spaltenreihenfolge und verbinden Sie beide Quellen zeilenweise.

Prüfen Sie danach, wie viele Beobachtungen aus jeder Quelle stammen.

> **Hinweis:** Fügen Sie die Quellenkennung vor `concat` hinzu, nicht erst danach.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: CSV und JSON laden und Herkunft erhalten
#
# Ziel dieser Codezelle:
# Laden Sie csvsourcea mit StringIO und jsonsourceb mit json.loads beziehungsweise
# pd.DataFrame. Ergänzen Sie in jeder Tabelle eine Spalte sourcefile,
# vereinheitlichen Sie die Spaltenreihenfolge und verbinden Sie beide...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# StringIO verhält sich für pandas wie eine kleine geöffnete Textdatei.
table_a = pd.read_csv(StringIO(csv_source_a))
table_a["source_file"] = "sensor_export_A.csv"

# json.loads wandelt die JSON-Zeichenkette in Python-Dictionaries um.
# Der DataFrame übernimmt deren Schlüssel als Spaltennamen.
records_b = json.loads(json_source_b)
table_b = pd.DataFrame(records_b)
table_b["source_file"] = "maintenance_snapshot_B.json"

# Eine feste Reihenfolge erleichtert die Kontrolle vor der Verkettung.
ordered_columns = [
    "machine_id",
    "temperature_c",
    "vibration_mm_s",
    "shift",
    "failure",
    "source_file",
]
table_a = table_a[ordered_columns]
table_b = table_b[ordered_columns]

# ignore_index=True erzeugt nach dem Zusammenfügen einen fortlaufenden Index.
combined_data = pd.concat([table_a, table_b], ignore_index=True)

source_counts = combined_data["source_file"].value_counts()

print(combined_data.to_string(index=False))
print("\nBeobachtungen je Quelle:")
print(source_counts.to_string())

### Reflexion zu Aufgabe 2

Die zusätzliche Quellspalte ermöglicht es, Qualitätsprobleme später zur Ursprungsdatei zurückzuverfolgen. Ohne diese Information wäre nach dem Zusammenfügen nicht mehr erkennbar, ob beispielsweise ein Wertebereichsproblem nur in einer Quelle vorkommt. Unterschiedliche Formate ändern nichts an der fachlichen Tabellenstruktur, solange Spaltenbedeutungen vereinheitlicht werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Zielwert und Merkmalstypen dokumentieren

    Erstellen Sie für die kombinierte Tabelle ein Datenwörterbuch mit mindestens folgenden Spalten:

- `column`
- `role` (`identifier`, `feature` oder `target`)
- `data_kind` (`numeric`, `categorical` oder `binary`)
- `unit`
- `expected_range_or_values`
- `description`

Ordnen Sie außerdem ein, ob `failure` eine Klassifikations- oder Regressionsaufgabe definiert.

> **Hinweis:** Ein als Zahl gespeichertes Feld kann fachlich trotzdem kategorial sein.

In [ ]:
# Falls Aufgabe 2 noch nicht ausgeführt wurde, können Sie die Daten hier erneut laden.
table_a = pd.read_csv(StringIO(csv_source_a)).assign(source_file="sensor_export_A.csv")
table_b = pd.DataFrame(json.loads(json_source_b)).assign(source_file="maintenance_snapshot_B.json")
combined_data = pd.concat([table_a, table_b], ignore_index=True)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Zielwert und Merkmalstypen dokumentieren
#
# Ziel dieser Codezelle:
# Erstellen Sie für die kombinierte Tabelle ein Datenwörterbuch mit mindestens
# folgenden Spalten: - column - role (identifier, feature oder target) - datakind
# (numeric, categorical oder binary) - unit - expectedrangeorv...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Falls Aufgabe 2 unabhängig ausgeführt wird, werden die beiden Quellen
# reproduzierbar noch einmal geladen und zusammengeführt.
table_a = pd.read_csv(StringIO(csv_source_a)).assign(
    source_file="sensor_export_A.csv"
)
table_b = pd.DataFrame(json.loads(json_source_b)).assign(
    source_file="maintenance_snapshot_B.json"
)
combined_data = pd.concat([table_a, table_b], ignore_index=True)

# Das Datenwörterbuch dokumentiert nicht nur technische Typen, sondern
# vor allem die fachliche Bedeutung und zulässige Werte.
data_dictionary = pd.DataFrame(
    [
        {
            "column": "machine_id",
            "role": "identifier",
            "data_kind": "categorical",
            "unit": "keine",
            "expected_range_or_values": "eindeutige Kennung",
            "description": "Identifiziert eine Maschine",
        },
        {
            "column": "temperature_c",
            "role": "feature",
            "data_kind": "numeric",
            "unit": "°C",
            "expected_range_or_values": "0 bis 120",
            "description": "Gemessene Betriebstemperatur",
        },
        {
            "column": "vibration_mm_s",
            "role": "feature",
            "data_kind": "numeric",
            "unit": "mm/s",
            "expected_range_or_values": "0 bis 20",
            "description": "Schwingungsgeschwindigkeit",
        },
        {
            "column": "shift",
            "role": "feature",
            "data_kind": "categorical",
            "unit": "keine",
            "expected_range_or_values": "Tag, Nacht",
            "description": "Arbeitsschicht der Messung",
        },
        {
            "column": "failure",
            "role": "target",
            "data_kind": "binary",
            "unit": "keine",
            "expected_range_or_values": "0 oder 1",
            "description": "Bestätigter Ausfallstatus",
        },
        {
            "column": "source_file",
            "role": "identifier",
            "data_kind": "categorical",
            "unit": "keine",
            "expected_range_or_values": "bekannte Quelldatei",
            "description": "Herkunft der Beobachtung",
        },
    ]
)

print(data_dictionary.to_string(index=False))

### Reflexion zu Aufgabe 3

`failure` ist ein binärer kategorialer Zielwert und definiert daher eine Klassifikationsaufgabe. Die numerische Speicherung als `0` und `1` macht den Zielwert nicht automatisch zu einer Regression. Das Datenwörterbuch hilft, technische Datentypen von fachlichen Rollen zu trennen und liefert Regeln für die spätere Validierung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Fehlwerte, Duplikate und Wertebereiche prüfen

    Prüfen Sie die kombinierte Tabelle auf:

1. fehlende Werte je Spalte,
2. vollständige Duplikate,
3. doppelte `machine_id`-Werte,
4. Temperaturen außerhalb 0 bis 120 °C,
5. negative Schwingungswerte,
6. unerwartete Schichtbezeichnungen.

Erstellen Sie eine Tabelle `issue_rows`, die alle Beobachtungen enthält, für die mindestens eine Prüfung fehlschlägt, und ergänzen Sie boolesche Prüfspalten.

> **Hinweis:** Formulieren Sie jede Qualitätsregel als eigene boolesche Spalte.

In [ ]:
table_a = pd.read_csv(StringIO(csv_source_a)).assign(source_file="sensor_export_A.csv")
table_b = pd.DataFrame(json.loads(json_source_b)).assign(source_file="maintenance_snapshot_B.json")
combined_data = pd.concat([table_a, table_b], ignore_index=True)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Fehlwerte, Duplikate und Wertebereiche prüfen
#
# Ziel dieser Codezelle:
# Prüfen Sie die kombinierte Tabelle auf: 1. fehlende Werte je Spalte, 2.
# vollständige Duplikate, 3. doppelte machineid-Werte, 4. Temperaturen außerhalb 0
# bis 120 °C, 5. negative Schwingungswerte, 6. unerwartete Schicht...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

table_a = pd.read_csv(StringIO(csv_source_a)).assign(
    source_file="sensor_export_A.csv"
)
table_b = pd.DataFrame(json.loads(json_source_b)).assign(
    source_file="maintenance_snapshot_B.json"
)
combined_data = pd.concat([table_a, table_b], ignore_index=True)

# Allgemeine Übersichten werden getrennt von den zeilenbezogenen
# Prüfspalten berechnet.
missing_counts = combined_data.isna().sum()
full_duplicate_count = int(combined_data.duplicated().sum())
duplicate_id_count = int(
    combined_data["machine_id"].duplicated(keep=False).sum()
)

checked = combined_data.copy()

# Jede boolesche Spalte dokumentiert genau eine fachliche Regel.
checked["missing_required_value"] = checked[
    ["machine_id", "temperature_c", "vibration_mm_s", "shift", "failure"]
].isna().any(axis=1)
checked["duplicate_machine_id"] = checked["machine_id"].duplicated(
    keep=False
)
checked["temperature_out_of_range"] = ~checked["temperature_c"].between(
    0, 120, inclusive="both"
) & checked["temperature_c"].notna()
checked["negative_vibration"] = (
    checked["vibration_mm_s"] < 0
) & checked["vibration_mm_s"].notna()
checked["unexpected_shift"] = ~checked["shift"].isin(["Tag", "Nacht"])

issue_columns = [
    "missing_required_value",
    "duplicate_machine_id",
    "temperature_out_of_range",
    "negative_vibration",
    "unexpected_shift",
]

# any(axis=1) markiert Zeilen, in denen mindestens eine Regel verletzt ist.
issue_rows = checked.loc[checked[issue_columns].any(axis=1)].copy()

print("Fehlwerte je Spalte:")
print(missing_counts.to_string())
print("Vollständige Duplikate:", full_duplicate_count)
print("Zeilen mit mehrfacher machine_id:", duplicate_id_count)
print("\nAuffällige Beobachtungen:")
print(issue_rows.to_string(index=False))

### Reflexion zu Aufgabe 4

Unterschiedliche Prüfungen beantworten unterschiedliche Fragen. Eine vollständige Duplikatzeile kann ein Exportfehler sein, während eine doppelte Maschinenkennung bei Messreihen durchaus legitim sein könnte. In diesem kleinen Snapshot wurde jedoch eine Beobachtung exakt dupliziert. Werte außerhalb der erwarteten Bereiche sollten zunächst markiert und fachlich geprüft werden, statt sie automatisch zu löschen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Wiederholbarer Validierungsbericht

    Schreiben Sie eine Funktion `validate_machine_data(df)`, die einen kompakten Bericht mit einer Zeile je Spalte zurückgibt. Der Bericht soll enthalten:

- Datentyp,
- Anzahl und Anteil fehlender Werte,
- Anzahl eindeutiger Werte,
- Minimum und Maximum für numerische Spalten,
- eine Textspalte `warning` mit relevanten Hinweisen.

Geben Sie zusätzlich ein zweites Dictionary mit Tabellenebenen-Kennzahlen zurück, darunter Zeilenzahl, Spaltenzahl, vollständige Duplikate und Anzahl eindeutiger Maschinen. Wenden Sie die Funktion auf die kombinierte Tabelle an und zeigen Sie auch `source_metadata`.

> **Hinweis:** Ein guter Bericht nennt konkrete Befunde und vermeidet automatische, nicht begründete Korrekturen.

In [ ]:
table_a = pd.read_csv(StringIO(csv_source_a)).assign(source_file="sensor_export_A.csv")
table_b = pd.DataFrame(json.loads(json_source_b)).assign(source_file="maintenance_snapshot_B.json")
combined_data = pd.concat([table_a, table_b], ignore_index=True)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Wiederholbarer Validierungsbericht
#
# Ziel dieser Codezelle:
# Schreiben Sie eine Funktion validatemachinedata(df), die einen kompakten Bericht
# mit einer Zeile je Spalte zurückgibt. Der Bericht soll enthalten: - Datentyp, -
# Anzahl und Anteil fehlender Werte, - Anzahl eindeutiger...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

table_a = pd.read_csv(StringIO(csv_source_a)).assign(
    source_file="sensor_export_A.csv"
)
table_b = pd.DataFrame(json.loads(json_source_b)).assign(
    source_file="maintenance_snapshot_B.json"
)
combined_data = pd.concat([table_a, table_b], ignore_index=True)

def validate_machine_data(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    # Erstellt reproduzierbare Spalten- und Tabellenprüfungen.

    rows = []

    # Jede Spalte wird mit derselben Grundlogik geprüft. Zusätzliche
    # fachliche Warnungen werden anschließend gezielt ergänzt.
    for column in df.columns:
        series = df[column]
        is_numeric = pd.api.types.is_numeric_dtype(series)

        minimum = series.min(skipna=True) if is_numeric else np.nan
        maximum = series.max(skipna=True) if is_numeric else np.nan
        warnings_for_column = []

        missing_count = int(series.isna().sum())
        if missing_count > 0:
            warnings_for_column.append("enthält Fehlwerte")

        if column == "temperature_c":
            if ((series < 0) | (series > 120)).fillna(False).any():
                warnings_for_column.append("Temperatur außerhalb 0..120")
        elif column == "vibration_mm_s":
            if (series < 0).fillna(False).any():
                warnings_for_column.append("negative Schwingung")
        elif column == "shift":
            if (~series.isin(["Tag", "Nacht"])).any():
                warnings_for_column.append("unerwartete Kategorie")
        elif column == "failure":
            if (~series.isin([0, 1])).any():
                warnings_for_column.append("Zielwert nicht binär")

        rows.append(
            {
                "column": column,
                "dtype": str(series.dtype),
                "missing_count": missing_count,
                "missing_share": missing_count / len(df),
                "unique_count": int(series.nunique(dropna=True)),
                "min": minimum,
                "max": maximum,
                "warning": "; ".join(warnings_for_column) or "keine",
            }
        )

    column_report = pd.DataFrame(rows)

    # Tabellenebenen-Kennzahlen beschreiben Struktur und Redundanz.
    table_report = {
        "row_count": int(df.shape[0]),
        "column_count": int(df.shape[1]),
        "full_duplicate_count": int(df.duplicated().sum()),
        "unique_machine_count": int(df["machine_id"].nunique()),
    }

    return column_report, table_report

validation_report, table_summary = validate_machine_data(combined_data)

print("Spaltenbericht:")
print(validation_report.round(3).to_string(index=False))
print("\nTabellenbericht:")
print(table_summary)
print("\nQuellenmetadaten:")
print(source_metadata.to_string(index=False))

### Reflexion zu Aufgabe 5

Die Funktion trennt allgemeine technische Kennzahlen von fachlichen Warnregeln. Dadurch kann sie wiederholt auf neue Versionen angewendet und zwischen Datenständen verglichen werden. Der Bericht ersetzt keine Bereinigungsentscheidung. Er zeigt vielmehr, welche Punkte vor Modellierung geklärt werden müssen. Quellenmetadaten, Version und Nutzungsberechtigung gehören zur Reproduzierbarkeit ebenso wie der eigentliche Tabelleninhalt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.